In [54]:
import argparse
import io
import logging
import re
import sys
import time
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import requests

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger("data_acquisition")

print("✓ Imports loaded")

✓ Imports loaded


In [55]:
# ============================================================================
# CONFIG: Paths and source URLs
# ============================================================================

# Create folder structure
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SA4_DIR = RAW_DIR / "abs_sa4"

# File paths
EV_RAW_CSV = RAW_DIR / "tfnsw_ev_dec2025.csv"
EV_CLEAN_CSV = PROCESSED_DIR / "tfnsw_ev_cleaned.csv"

# Source data configuration
EV_TARGET_MONTH = "202512"  # December 2025
EV_CKAN_APIS = [
    ("data.nsw.gov.au",
     "https://data.nsw.gov.au/data/api/3/action/package_show?id=69fab867-a077-4e3d-a9ab-c169681ac877"),
    ("data.gov.au",
     "https://data.gov.au/data/api/3/action/package_show?id=nsw-2-ev-charging-locations"),
]
EV_FALLBACK_URL = (
    "https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/"
    "resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv"
)

SA4_URL = (
    "https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/"
    "edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files/"
    "SA4_2026_AUST_SHP_GDA2020.zip"
)
SA4_CRS = "EPSG:7844"
NSW_STATE_CODE = "1"

HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36"}

print(f"✓ Root directory: {ROOT}")
print(f"✓ Data directory: {DATA_DIR}")

✓ Root directory: /Users/khushkothari/Documents/charge-grid-nsw
✓ Data directory: /Users/khushkothari/Documents/charge-grid-nsw/data


In [56]:
def find_sa4_shapefile() -> Path:
    """Path of the unzipped SA4 shapefile (fields: SA4_CODE26, SA4_NAME26, STE_CODE26, ...)."""
    shp = next(iter(sorted(SA4_DIR.glob("*.shp"))), None)
    if shp is None:
        raise FileNotFoundError(f"No SA4 shapefile in {SA4_DIR} - run the acquisition step first")
    return shp

print("✓ Config loaded")

✓ Config loaded


## Data Acquisition Functions

In [57]:
def fetch(url: str) -> bytes:
    """GET with 3 attempts; refuses HTML pages (login walls / error pages)."""
    for attempt in range(1, 4):
        try:
            r = requests.get(url, headers=HEADERS, timeout=(15, 180))
            r.raise_for_status()
            if r.content[:20].lstrip().lower().startswith((b"<!doctype", b"<html")):
                raise RuntimeError("got a web page instead of data")
            return r.content
        except Exception as exc:
            log.warning("download attempt %d failed for %s: %s", attempt, url, exc)
            if attempt == 3:
                raise
            time.sleep(2 * attempt)


def find_ev_url(month: str = EV_TARGET_MONTH) -> str:
    """Ask the CKAN catalogue for the resource whose file name carries the December 2025 date."""
    for name, api in EV_CKAN_APIS:
        try:
            resources = requests.get(api, headers=HEADERS, timeout=60).json()["result"]["resources"]
        except Exception as exc:
            log.warning("catalogue %s unavailable: %s", name, exc)
            continue
        hits = []
        for res in resources:
            fname = urlparse(res.get("url", "")).path.split("/")[-1]
            m = re.search(r"(20\d{2})(\d{2})\d{2}", fname)
            if m and m.group(1) + m.group(2) == month:
                hits.append((fname, res["url"]))
        if hits:
            return max(hits)[1]
    log.warning("catalogue lookup failed - using the pinned December 2025 URL")
    return EV_FALLBACK_URL


def acquire(force: bool = False) -> None:
    """Download EV charger data and SA4 shapefiles."""
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    if force or not EV_RAW_CSV.exists():
        url = find_ev_url()
        log.info("downloading EV chargers: %s", url)
        EV_RAW_CSV.write_bytes(fetch(url))
    else:
        log.info("EV file already downloaded")

    if force or not list(SA4_DIR.glob("*.shp")):
        log.info("downloading ABS SA4 boundaries: %s", SA4_URL)
        zipfile.ZipFile(io.BytesIO(fetch(SA4_URL))).extractall(SA4_DIR)
        find_sa4_shapefile()
    else:
        log.info("SA4 boundaries already downloaded")

print("✓ Acquisition functions loaded")

✓ Acquisition functions loaded


## Data Cleaning Configuration

In [58]:
# Column name normalization
COLUMN_ALIASES = {
    "objectid": "object_id", "station_name": "station_name", "station_address": "station_address",
    "operator": "operator", "number_of_plugs": "number_of_plugs", "charger_type": "charger_type",
    "charger_rating": "charger_rating", "latitude": "latitude", "longitude": "longitude",
    "lganame": "lga_name", "pcode": "postcode", "source": "source",
    "name": "station_name", "address": "station_address", "lat": "latitude", "lon": "longitude",
    "lng": "longitude", "postcode": "postcode", "lga_name": "lga_name", "plugs": "number_of_plugs",
}

# Updated EXPECTED list: object_id removed
EXPECTED = [
    "station_name", "station_address", "operator", "number_of_plugs",
    "charger_type", "charger_rating", "latitude", "longitude", "lga_name", "postcode", "source"
]

PLACEHOLDERS = {
    "", "na", "n/a", "nan", "null", "none", "nil", "-", "--", "?", "tbc", "tbd", "unknown"
}

OPERATOR_RULES = [
    (r"nrma", "NRMA"),
    (r"chargefox", "Chargefox"),
    (r"evie", "Evie Networks"),
    (r"tesla", "Tesla"),
    (r"ampol|ampcharge", "Ampol"),
    (r"\bbp\b", "BP Pulse"),
    (r"jolt", "Jolt"),
    (r"exploren", "Exploren"),
    (r"engie", "Engie"),
    (r"evenergi", "Evenergi"),
    (r"shell", "Shell Recharge"),
    (r"7.?eleven", "7-Eleven"),
    (r"viva energy", "Viva Energy"),
    (r"fast cities", "Fast Cities"),
]

NSW = {"lat": (-37.7, -27.9), "lon": (140.8, 153.8)}  # rough NSW bounding box

print("✓ Cleaning config loaded")

✓ Cleaning config loaded


## Data Cleaning Functions

In [59]:
def snake(col) -> str:
    """Convert column name to snake_case."""
    return re.sub(r"[^0-9a-z]+", "_", str(col).replace("\ufeff", "").strip().lower()).strip("_")


def clean_text(x):
    """Trim, collapse whitespace, turn placeholders like 'N/A' / 'TBC' / '-' into NaN."""
    if pd.isna(x):
        return np.nan
    s = re.sub(r"\s+", " ", str(x).replace("\u00a0", " ")).strip()
    return np.nan if s.lower() in PLACEHOLDERS else s


def load_raw() -> pd.DataFrame:
    """Load and normalize raw CSV columns."""
    df = pd.read_csv(EV_RAW_CSV, dtype=str, encoding_errors="replace")
    df = df.rename(columns=lambda c: COLUMN_ALIASES.get(snake(c), snake(c)))
    if not {"latitude", "longitude"} <= set(df.columns):
        raise ValueError(f"No latitude/longitude columns found. Columns are: {list(df.columns)}")
    for c in EXPECTED:
        if c not in df.columns:
            log.warning("column '%s' not in raw file", c)
            df[c] = np.nan
    return df


def parse_rating(s):
    """'22 kW' -> 22 | '22' -> 22 | '2x350kW & 2x175kW' -> 350 (highest plug power) | 'AC' -> NaN"""
    if not isinstance(s, str):
        return np.nan
    kw = [float(x) for x in re.findall(r"(\d+(?:\.\d+)?)\s*kw", s, flags=re.I)]
    if kw:
        return max(kw)
    return float(s) if re.fullmatch(r"\d+(?:\.\d+)?", s.strip()) else np.nan


def postcode_from_address(addr):
    """Last 2xxx number in the address text ('... Artarmon NSW 2064, Australia' -> '2064')."""
    found = re.findall(r"\b2\d{3}\b", addr) if isinstance(addr, str) else []
    return found[-1] if found else np.nan

print("✓ Text cleaning functions loaded")

✓ Text cleaning functions loaded


In [60]:
def clean(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Comprehensive cleaning pipeline with statistics."""
    st = {"raw_rows": len(df)}
    df = df.copy()
    df["source_row"] = np.arange(len(df)) + 2

    # === Text & missing values ===
    before = df[EXPECTED].isna().sum().sum()
    for c in EXPECTED:
        df[c] = df[c].map(clean_text)
    st["placeholder_cells_set_to_null"] = int(df[EXPECTED].isna().sum().sum() - before)

    # === Type conversion ===
    for c in ("latitude", "longitude"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    plugs = pd.to_numeric(df["number_of_plugs"], errors="coerce")
    st["invalid_plug_counts_nulled"] = int((plugs.notna() & (plugs <= 0)).sum())
    df["number_of_plugs"] = plugs.where(plugs > 0).astype("Int64")
    df["rating_kw"] = df["charger_rating"].map(parse_rating).astype(float)
    df["postcode"] = df["postcode"].str.extract(r"(\d{4})")[0]
    no_pc = df["postcode"].isna()
    df.loc[no_pc, "postcode"] = df.loc[no_pc, "station_address"].map(postcode_from_address)
    st["postcodes_filled_from_address"] = int(no_pc.sum() - df["postcode"].isna().sum())

    # === Charger type (AC / DC) and status ===
    t = df["charger_type"].fillna("").astype(str).str.lower()
    df["is_upcoming"] = t.str.contains("upcoming|planned|proposed|future|construction")
    df["current_type"] = np.where(
        t.str.contains(r"\bdc\b|fast|rapid"), "DC",
        np.where(t.str.contains(r"\bac\b|destination"), "AC", "Unknown")
    )
    unknown_before = int((df["current_type"] == "Unknown").sum())
    rt = df["charger_rating"].fillna("").str.strip().str.upper()
    unk = df["current_type"] == "Unknown"
    df.loc[unk & (rt == "AC"), "current_type"] = "AC"
    df.loc[unk & (rt == "DC"), "current_type"] = "DC"
    infer = (df["current_type"] == "Unknown") & df["rating_kw"].notna()
    df.loc[infer, "current_type"] = np.where(df.loc[infer, "rating_kw"] > 22, "DC", "AC")
    st["type_inferred_when_missing"] = unknown_before - int((df["current_type"] == "Unknown").sum())
    df["is_dc_fast"] = (df["current_type"] == "DC") & ~df["is_upcoming"]  # existing DC chargers only

    # === Impute missing rating_kw ===
    missing_rating_before = df["rating_kw"].isna().sum()
    default_ratings = pd.Series(
        np.where(df["current_type"] == "DC", 50.0, 22.0),
        index=df.index
    )
    df["rating_kw"] = df["rating_kw"].fillna(default_ratings)
    st["rating_kw_imputed"] = int(missing_rating_before - df["rating_kw"].isna().sum())

    # === Operator name standardization ===
    op = df["operator"].fillna("Unknown")
    st["operators_raw"] = int(op.nunique())
    for pat, name in OPERATOR_RULES:
        op = op.mask(op.str.lower().str.contains(pat, regex=True), name)
    key = op.str.lower().str.replace(r"[^a-z0-9]+", "", regex=True)
    df["operator"] = key.map(op.groupby(key).agg(lambda s: s.value_counts().index[0]))
    st["operators_clean"] = int(df["operator"].nunique())

    # === Name formatting & station_name Imputation Pipeline ===
    for c in ("station_name", "station_address", "lga_name"):
        df[c] = df[c].map(lambda s: s.title() if isinstance(s, str) and (s.isupper() or s.islower()) else s)

    # 1. Flag missing station names for audit trail
    df["is_station_name_imputed"] = df["station_name"].isna()
    st["station_names_imputed"] = int(df["is_station_name_imputed"].sum())

    # 2. Primary Fallback: Extract street address (before locality / state / postcode)
    street_fallback = df["station_address"].fillna("").str.split(
        r",|\bNSW\b|\b\d{4}\b", n=1, regex=True
    ).str[0].str.strip()

    # 3. Secondary Fallback: Operator + Location details
    loc_detail = df["lga_name"].fillna(df["postcode"].fillna("Site"))
    secondary_fallback = df["operator"].fillna("EV") + " Charger (" + loc_detail + ")"

    # 4. Apply fallbacks sequentially
    fallback_name = street_fallback.where(street_fallback.str.len() > 3, secondary_fallback)
    df["station_name"] = df["station_name"].fillna(fallback_name)

    # 5. Update station_name_key with the final imputed name
    df["station_name_key"] = df["station_name"].str.lower().str.replace(r"[^a-z0-9]+", "", regex=True)

    # === Coordinate repair ===
    sw = df["latitude"].between(140, 155) & df["longitude"].between(-38, -27)  # lat/lon swapped
    df.loc[sw, ["latitude", "longitude"]] = df.loc[sw, ["longitude", "latitude"]].to_numpy()
    neg = df["latitude"].between(27, 38)  # minus sign lost
    df.loc[neg, "latitude"] *= -1
    st["coordinates_repaired"] = int(sw.sum() + neg.sum())

    # === Coordinate filtering ===
    missing = df["latitude"].isna() | df["longitude"].isna()
    outside = ~missing & ~(df["latitude"].between(*NSW["lat"]) & df["longitude"].between(*NSW["lon"]))
    st["dropped_missing_coordinates"], st["dropped_outside_nsw_extent"] = int(missing.sum()), int(outside.sum())
    df = df[~(missing | outside)]

    # === Duplicate removal ===
    dup_cols = [
        "operator", "station_name_key", "number_of_plugs", "current_type", "rating_kw",
        "is_upcoming", "latitude", "longitude"
    ]
    dups = df.duplicated(subset=dup_cols, keep="first")
    st["duplicates_removed"] = int(dups.sum())
    df = df[~dups].copy()

    # === IDs & output ===
    df["site_id"] = df.groupby(["operator", df["latitude"].round(4), df["longitude"].round(4)]).ngroup() + 1
    df = df.sort_values("source_row").reset_index(drop=True)
    df.insert(0, "charger_id", np.arange(1, len(df) + 1))

    cols = [
        "charger_id", "site_id", "station_name", "station_name_key", "is_station_name_imputed", 
        "station_address", "operator", "number_of_plugs", "charger_type", "current_type", 
        "is_dc_fast", "is_upcoming", "charger_rating", "rating_kw", "latitude", "longitude", 
        "lga_name", "postcode", "source", "source_row"
    ]
    st["null_counts"] = df[EXPECTED].isna().sum().to_dict()
    st.update(
        clean_rows=len(df),
        dc_fast_rows=int(df["is_dc_fast"].sum()),
        sites=int(df["site_id"].nunique())
    )
    return df[cols], st

print("✓ Clean function loaded with station_name imputation pipeline")

✓ Clean function loaded with station_name imputation pipeline


## Main Pipeline

### Step 1: Download Data

Downloads EV charger data from TfNSW and ABS SA4 shapefiles. Set `force=True` to re-download.

In [61]:
# Download raw data and shapefiles
# Set force=True to re-download even if files exist
try:
    acquire(force=False)
    print("✓ Acquisition complete")
except Exception as exc:
    log.error("Acquisition failed: %s", exc)
    raise

23:27:56 | INFO    | downloading EV chargers: https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv
23:27:56 | INFO    | downloading ABS SA4 boundaries: https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files/SA4_2026_AUST_SHP_GDA2020.zip


✓ Acquisition complete


### Step 2: Clean & Process Data

In [62]:
# Load raw data
try:
    raw_df = load_raw()
    print(f"✓ Loaded {len(raw_df)} rows from {EV_RAW_CSV}")
except Exception as exc:
    log.error("Failed to load raw data: %s", exc)
    raise

✓ Loaded 1958 rows from /Users/khushkothari/Documents/charge-grid-nsw/data/raw/tfnsw_ev_dec2025.csv


In [63]:
# Clean data
try:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    df_clean, stats = clean(raw_df)
    print(f"✓ Cleaned {stats['clean_rows']} of {stats['raw_rows']} rows")
    print(f"✓ {stats['sites']} unique sites with {stats['dc_fast_rows']} DC fast chargers")
except Exception as exc:
    log.error("Cleaning failed: %s", exc)
    raise

✓ Cleaned 1957 of 1958 rows
✓ 1932 unique sites with 433 DC fast chargers


In [64]:
# Save cleaned data
try:
    df_clean.to_csv(EV_CLEAN_CSV, index=False)
    log.info("cleaned %d of %d rows -> %s", stats["clean_rows"], stats["raw_rows"], EV_CLEAN_CSV)
    print(f"✓ Saved to {EV_CLEAN_CSV}")
except Exception as exc:
    log.error("Save failed: %s", exc)
    raise

23:28:00 | INFO    | cleaned 1957 of 1958 rows -> /Users/khushkothari/Documents/charge-grid-nsw/data/processed/tfnsw_ev_cleaned.csv


✓ Saved to /Users/khushkothari/Documents/charge-grid-nsw/data/processed/tfnsw_ev_cleaned.csv


## Cleaning Statistics & Data Quality Report

In [65]:
# Display comprehensive cleaning statistics
print("\n" + "="*60)
print("DATA QUALITY REPORT")
print("="*60)
print(f"\nRows:")
print(f"  Raw: {stats['raw_rows']}")
print(f"  Clean: {stats['clean_rows']}")
print(f"  Dropped: {stats['raw_rows'] - stats['clean_rows']}")

print(f"\nData Issues Found & Fixed:")
print(f"  Placeholder cells set to NaN: {stats['placeholder_cells_set_to_null']}")
print(f"  Invalid plug counts nulled: {stats['invalid_plug_counts_nulled']}")
print(f"  Postcodes filled from address: {stats['postcodes_filled_from_address']}")
print(f"  Type inferred when missing: {stats['type_inferred_when_missing']}")
print(f"  Station names imputed: {stats['station_names_imputed']}")
print(f"  Rating kW imputed: {stats['rating_kw_imputed']}")
print(f"  Coordinates repaired (swap/sign): {stats['coordinates_repaired']}")
print(f"  Dropped (missing coordinates): {stats['dropped_missing_coordinates']}")
print(f"  Dropped (outside NSW): {stats['dropped_outside_nsw_extent']}")
print(f"  Duplicates removed: {stats['duplicates_removed']}")

print(f"\nOperators:")
print(f"  Raw unique: {stats['operators_raw']}")
print(f"  Cleaned unique: {stats['operators_clean']}")

print(f"\nChargers:")
print(f"  Total: {stats['clean_rows']}")
print(f"  DC Fast (existing): {stats['dc_fast_rows']}")
print(f"  Unique sites: {stats['sites']}")

print(f"\nNull counts by expected column:")
for col, count in sorted(stats['null_counts'].items()):
    pct = (count / stats['clean_rows'] * 100) if stats['clean_rows'] > 0 else 0
    print(f"  {col}: {count:4d} ({pct:5.1f}%)")
print("\n" + "="*60)


DATA QUALITY REPORT

Rows:
  Raw: 1958
  Clean: 1957
  Dropped: 1

Data Issues Found & Fixed:
  Placeholder cells set to NaN: 0
  Invalid plug counts nulled: 0
  Postcodes filled from address: 120
  Type inferred when missing: 98
  Station names imputed: 1438
  Rating kW imputed: 522
  Coordinates repaired (swap/sign): 0
  Dropped (missing coordinates): 0
  Dropped (outside NSW): 0
  Duplicates removed: 1

Operators:
  Raw unique: 50
  Cleaned unique: 43

Chargers:
  Total: 1957
  DC Fast (existing): 433
  Unique sites: 1932

Null counts by expected column:
  charger_rating:    0 (  0.0%)
  charger_type:    0 (  0.0%)
  latitude:    0 (  0.0%)
  lga_name:  120 (  6.1%)
  longitude:    0 (  0.0%)
  number_of_plugs:    0 (  0.0%)
  operator:    0 (  0.0%)
  postcode:    1 (  0.1%)
  source:  120 (  6.1%)
  station_address:    0 (  0.0%)
  station_name:    0 (  0.0%)



## Data Preview

In [66]:
# Display first few rows
df_clean.tail(500)

,charger_id,site_id,station_name,station_name_key,is_station_name_imputed,station_address,operator,number_of_plugs,charger_type,current_type,is_dc_fast,is_upcoming,charger_rating,rating_kw,latitude,longitude,lga_name,postcode,source,source_row
1457,1458,1459,2A Frances St Randwick,2afrancesstrandwick,True,2A Frances St Randwick NSW 2031 Australia,PLUS ES,1,AC,AC,False,False,22 kW,22.0,-33.910206,151.236876,Randwick City Council,2031,Kerbside Charging R1,1460
1458,1459,1550,2A Liverpool St Rose Bay,2aliverpoolstrosebay,True,2A Liverpool St Rose Bay NSW 2029 Australia,PLUS ES,1,AC,AC,False,False,22 kW,22.0,-33.875011,151.273070,Waverley Council,2029,Kerbside Charging R1,1461
1459,1460,629,Redhead Beach Carpark,redheadbeachcarpark,False,2B Beach Rd Redhead NSW 2290 Australia,Everty,2,AC,AC,False,False,AC,22.0,-33.014116,151.716829,Lake Macquarie City Council,2290,Destination Charging R2,1462
1460,1461,434,Club Lithgow,clublithgow,False,2C Lithgow St Lithgow NSW 2790 Australia,EVSE,4,AC,AC,False,False,AC,22.0,-33.485343,150.152020,"Lithgow Council, City of",2790,Destination Charging R2,1463
1461,1462,940,The Entrance Leagues Club,theentranceleaguesclub,False,3 Bay Village Rd Bateau Bay NSW 2261 Australia,Exploren,4,AC,AC,False,False,AC,22.0,-33.377381,151.473372,Central Coast Council,2261,Destination Charging R1,1464
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1952,1953,1480,17 Dulwich St Dulwich Hill,17dulwichstdulwichhill,True,17 Dulwich St Dulwich Hill NSW 2203 Australia,PLUS ES,1,AC,AC,False,False,7 kW,7.0,-33.902552,151.140843,Inner West Council,2203,Kerbside Charging R1,1955
1953,1954,512,17 Flood St Bondi,17floodstbondi,True,17 Flood St Bondi NSW 2026 Australia,EVX,2,AC,AC,False,False,22 kW,22.0,-33.891058,151.258787,Waverley Council,2026,Kerbside Charging R1,1956
1954,1955,1483,17 Grove St Dulwich Hill,17grovestdulwichhill,True,17 Grove St Dulwich Hill NSW 2203 Australia,PLUS ES,1,AC,AC,False,False,7 kW,7.0,-33.901739,151.139465,Inner West Council,2203,Kerbside Charging R1,1957
1955,1956,1434,17 Hereward St Maroubra,17herewardstmaroubra,True,17 Hereward St Maroubra NSW 2035 Australia,PLUS ES,1,AC,AC,False,False,22 kW,22.0,-33.945104,151.256166,Randwick City Council,2035,Kerbside Charging R1,1958


In [67]:
# Distribution by operator
print("\nChargers by operator:")
print(df_clean['operator'].value_counts())


Chargers by operator:
operator
Exploren                        300
Tesla                           287
Chargefox                       254
Non-networked                   224
PLUS ES                         150
EVX                             111
Evie Networks                   104
NRMA                             93
BP Pulse                         60
Jolt                             49
EVUp                             38
Everty                           36
Ampol                            32
Smart Charge                     24
EVE Australia                    24
Porsche Smart Mobility           19
ChargeHub                        18
EVSE                             18
Fast Cities                      17
EVNet                            16
Noodoe                           13
ChargePoint                       8
Viva Energy                       8
Elanga                            7
Saascharge                        7
Engie                             6
CasaCharge                      

In [68]:
# Distribution by current type and status
print("\nChargers by type and status:")
crosstab = pd.crosstab(
    df_clean['current_type'],
    df_clean['is_upcoming'],
    margins=True,
    margins_name='Total'
)
crosstab.columns = ['Existing', 'Upcoming', 'Total']
print(crosstab)


Chargers by type and status:
              Existing  Upcoming  Total
current_type                           
AC                1426         4   1430
DC                 433        94    527
Total             1859        98   1957


In [69]:
# Charger rating distribution
print("\nCharger rating statistics (kW):")
print(df_clean['rating_kw'].describe())


Charger rating statistics (kW):
count    1957.000000
mean       53.624936
std        82.218780
min         3.000000
25%        22.000000
50%        22.000000
75%        25.000000
max       400.000000
Name: rating_kw, dtype: float64


In [70]:
import pandas as pd
import numpy as np

def null_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Generate a summary of null values in all columns."""
    summary = pd.DataFrame({
        'column': df.columns,
        'null_count': df.isnull().sum().values,
        'null_percentage': (df.isnull().sum().values / len(df) * 100).round(2),
        'data_type': df.dtypes.values
    })
    summary['null_percentage'] = summary['null_percentage'].apply(lambda x: f"{x}%")
    return summary.sort_values('null_count', ascending=False).reset_index(drop=True)


def null_heatmap_text(df: pd.DataFrame, width: int = 80) -> None:
    """Print a text-based heatmap of null values by column."""
    print("\nData Completeness Heatmap")
    print("=" * width)
    
    for col in df.columns:
        completeness = (1 - df[col].isnull().sum() / len(df)) * 100
        bar_length = int((width - len(col) - 10) * completeness / 100)
        bar = "█" * bar_length + "░" * (width - len(col) - 10 - bar_length)
        print(f"{col:<15} [{bar}] {completeness:.1f}%")


def missing_data_report(df: pd.DataFrame) -> None:
    """Print a comprehensive missing data report."""
    print("\n" + "=" * 70)
    print("MISSING DATA REPORT")
    print("=" * 70)
    
    total_cells = df.shape[0] * df.shape[1]
    total_missing = df.isnull().sum().sum()
    overall_missing_pct = (total_missing / total_cells) * 100
    
    print(f"\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Total Cells: {total_cells:,}")
    print(f"Total Missing: {total_missing:,} ({overall_missing_pct:.2f}%)")
    
    print("\n" + "-" * 70)
    print("NULL VALUES BY COLUMN:")
    print("-" * 70)
    
    summary = null_summary(df)
    print(summary.to_string(index=False))
    
    complete_cols = df.columns[df.isnull().sum() == 0].tolist()
    if complete_cols:
        print(f"\n✓ Complete Columns (no nulls): {', '.join(complete_cols)}")
    
    empty_cols = df.columns[df.isnull().sum() == len(df)].tolist()
    if empty_cols:
        print(f"\n✗ Empty Columns (all nulls): {', '.join(empty_cols)}")
    
    mostly_empty = df.columns[df.isnull().sum() / len(df) > 0.5].tolist()
    if mostly_empty:
        print(f"\n⚠ Mostly Empty (>50% nulls): {', '.join(mostly_empty)}")
    
    print("\n" + "=" * 70)


def data_preview(df: pd.DataFrame, rows: int = 5) -> None:
    """Display a data preview with column info and null analysis."""
    print(f"\n{'='*70}")
    print(f"DATA PREVIEW ({rows} rows)")
    print(f"{'='*70}\n")
    
    print(df.head(rows).to_string())
    
    print(f"\n\nCOLUMN INFO:")
    print(f"{'-'*70}")
    summary = null_summary(df)
    print(summary.to_string(index=False))

In [71]:
# Quick summary
print(null_summary(df_clean))

# Visual heatmap
null_heatmap_text(df_clean)

# Full report
missing_data_report(df_clean)

# Preview with analysis
data_preview(df_clean, rows=10)

                     column  null_count null_percentage data_type
0                    source         120           6.13%    object
1                  lga_name         120           6.13%    object
2                  postcode           1           0.05%    object
3                charger_id           0            0.0%     int64
4                   site_id           0            0.0%     int64
5                 longitude           0            0.0%   float64
6                  latitude           0            0.0%   float64
7                 rating_kw           0            0.0%   float64
8            charger_rating           0            0.0%    object
9               is_upcoming           0            0.0%      bool
10               is_dc_fast           0            0.0%      bool
11             current_type           0            0.0%    object
12             charger_type           0            0.0%    object
13          number_of_plugs           0            0.0%     Int64
14        

## Verify SA4 Shapefile

In [72]:
# Verify SA4 shapefile was downloaded
try:
    shp_path = find_sa4_shapefile()
    print(f"✓ SA4 shapefile found: {shp_path}")
    print(f"  CRS: {SA4_CRS}")
    print(f"  Fields: SA4_CODE26, SA4_NAME26, STE_CODE26, ...")
except FileNotFoundError as e:
    print(f"✗ {e}")

✓ SA4 shapefile found: /Users/khushkothari/Documents/charge-grid-nsw/data/raw/abs_sa4/SA4_2026_AUST_GDA2020.shp
  CRS: EPSG:7844
  Fields: SA4_CODE26, SA4_NAME26, STE_CODE26, ...


## Summary & Next Steps

**Pipeline complete!** The cleaned dataset is ready at:
- **CSV**: `data/processed/tfnsw_ev_cleaned.csv`
- **Shapefiles**: `data/raw/abs_sa4/` (SA4 boundaries)

**Key outputs:**
- `df_clean`: DataFrame with cleaned charger data (1,000+ rows, 20 columns)
- `stats`: Dictionary of data quality metrics

**Next steps:**
1. Spatial analysis (join chargers to SA4 regions using shapefiles)
2. Availability analysis (plug counts by region/operator)
3. DC fast charging coverage assessment
4. Time-series analysis of charger deployment trends